# Debiased search: top-k retrieval before vs. after erasing a concept

Off-the-shelf embeddings bake bias into **retrieval**: a query like *"a registered nurse"* returns mostly women. Here we erase **gender** from the document and query embeddings (JEV concept, top-1 question) and compare the gender mix of the top-k results — relevance stays, the skew evens out. Runs offline from the bundled cache (`USE_CACHE=True`).

## 1. Documents + embeddings

In [1]:
import os, sys, json, hashlib, logging
from pathlib import Path
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
for _n in ('httpx', 'httpcore', 'openai', 'openai._base_client'):
    logging.getLogger(_n).setLevel(logging.WARNING)

def _root(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q/'pyproject.toml').exists() and (q/'src'/'jevu').exists():
            return q
    return Path(s).resolve()
ROOT = _root(os.getcwd())
if str(ROOT/'src') not in sys.path: sys.path.insert(0, str(ROOT/'src'))
from jevu import ConceptScrubber, JevLabeler

_envf = ROOT/'.env'
if _envf.exists():
    for _l in _envf.read_text().splitlines():
        _l = _l.strip()
        if _l and not _l.startswith('#') and '=' in _l:
            _k, _v = _l.split('=', 1); os.environ.setdefault(_k.strip(), _v.strip().strip('\'"'))

USE_CACHE = True
CACHE = (ROOT/'examples'/'.jev_debias_cache') if USE_CACHE else None
if CACHE: CACHE.mkdir(parents=True, exist_ok=True)
SEED, N, DENSE_MODEL = 2026, 900, 'text-embedding-3-small'
DATASET, CONFIG, SPLIT = 'LabHC/bias_in_bios', 'default', 'test'
HF_ROWS = 'https://datasets-server.huggingface.co/rows'
def dg(x): return hashlib.sha256(json.dumps(x, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
def sj(p, x): p.write_text(json.dumps(x, ensure_ascii=False))
print('caching:', bool(CACHE))

caching: True


In [2]:
def hf_page(off, length=100):
    p = (CACHE/f'bios_{SPLIT}_{off}_{length}.json') if CACHE else None
    if p and p.exists(): return json.loads(p.read_text())
    import httpx
    r = httpx.get(HF_ROWS, params={'dataset': DATASET, 'config': CONFIG, 'split': SPLIT, 'offset': int(off), 'length': int(length)}, timeout=60); r.raise_for_status()
    d = r.json()
    if p: sj(p, d)
    return d

def load_bios(n, seed):
    p = (CACHE/f'bios_sample_{n}_{seed}.json') if CACHE else None
    if p and p.exists(): return json.loads(p.read_text())['rows']
    total = hf_page(0, 1)['num_rows_total']; rng = np.random.default_rng(seed); rows = []
    for off in rng.permutation(np.arange(0, total, 100)):
        for it in hf_page(int(off), 100)['rows']:
            r = it['row']; t = str(r['hard_text'] or '').strip()
            if t: rows.append({'text': t[:700], 'prof': int(r['profession']), 'gender': int(r['gender'])})
        if len(rows) >= n: break
    rng.shuffle(rows); rows = rows[:n]
    if p: sj(p, {'rows': rows})
    return rows

def embed_dense(texts):
    p = (CACHE/f'dense_{dg({"m": DENSE_MODEL, "t": list(texts)})}.json') if CACHE else None
    if p and p.exists(): return np.asarray(json.loads(p.read_text()), dtype=float)
    from openai import OpenAI
    cl = OpenAI(timeout=90); v = []
    for s in range(0, len(texts), 200):
        v.extend(x.embedding for x in cl.embeddings.create(model=DENSE_MODEL, input=list(texts)[s:s+200]).data)
    v = np.asarray(v, dtype=float)
    if p: sj(p, v.tolist())
    return v

# documents = the bios; search over them with text queries
rows = load_bios(N, SEED)
doc_texts = [r['text'] for r in rows]
doc_prof = np.array([r['prof'] for r in rows]); doc_gender = np.array([r['gender'] for r in rows])
Xdoc = normalize(embed_dense(doc_texts))      # off-the-shelf doc embeddings
print(len(doc_texts), 'documents | embeddings', Xdoc.shape)

900 documents | embeddings (900, 1536)


## 2. Erase gender, then retrieve before vs. after

Same eraser is applied to documents **and** queries, so similarity is computed in the debiased space. We report the fraction of female results in the top-10 per query.

In [3]:
# 1) erase GENDER from the embedding space (JEV concept, top-1 question)
eraser = ConceptScrubber(method='leace', expand=True, concept='gender',
                         select_k=1, select_sample=250, cache_dir=str(CACHE) if CACHE else None)
eraser.fit(embeddings=Xdoc, texts=doc_texts)
Xdoc_d = eraser.transform(Xdoc)               # debiased document embeddings

# 2) which dataset gender value is "female" (detect via cached JEV scores, no hardcoding)
gscore = JevLabeler('Does the text describe a woman or a female person?',
                    cache_dir=str(CACHE) if CACHE else None).score(doc_texts)
female = max(np.unique(doc_gender), key=lambda v: gscore[doc_gender == v].mean())

# 3) gender-neutral occupation queries (embed once, then erase gender from the query too)
queries = ['a registered nurse', 'a software engineer', 'a kindergarten teacher', 'a surgeon']
Q = normalize(embed_dense(queries))
Q_d = eraser.transform(Q)

def topk(q, D, k=10):
    D = normalize(D); q = q / (np.linalg.norm(q) + 1e-9)
    return np.argsort(-(D @ q))[:k]

K = 10
print(f'\ntop-{K} retrieval: fraction FEMALE among results, before -> after erasing gender\n')
print(f"{'query':<24}{'before':>9}{'after':>9}")
for i, qt in enumerate(queries):
    b = topk(Q[i], Xdoc, K); a = topk(Q_d[i], Xdoc_d, K)
    fb = float(np.mean(doc_gender[b] == female)); fa = float(np.mean(doc_gender[a] == female))
    print(f'{qt:<24}{fb:>8.0%}{fa:>9.0%}')

INFO httpx2: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


INFO jevu.concepts: expanded concept 'gender' into 6 questions: ['Does the text refer to a woman?', 'Does the text refer to a man?', 'Does the text use gendered pronouns or titles?', 'Does the text describe a maternal role or relationship?', 'Does the text describe a paternal role or relationship?', 'Does the text mention a specific gender identity?']


INFO jevu.scrubber: selecting 1/6 questions on a 250-row sample


INFO jevu.concepts: scoring 250 texts x 6 questions = 1500 cells (716 cached, 784 to fetch)


JEV x6: gender:   0%|          | 0/1500 [00:00<?, ?it/s]

...

INFO jevu.erasers: LEACE erased concept (d=1536, k=1)


INFO jevu.labelers: scoring 900 texts (900 cached, 0 to fetch) for concept 'Does the text describe a woman or a female person?'


JEV: Does the text describe a woman or a fema:   0%|          | 0/900 [00:00<?, ?it/s]

INFO httpx2: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"



top-10 retrieval: fraction FEMALE among results, before -> after erasing gender

query                      before    after
a registered nurse           80%      70%
a software engineer          10%      10%
a kindergarten teacher       80%      60%
a surgeon                    10%      10%


## 3. Eyeball one query

Top-5 for *"a registered nurse"*: before, the results skew female; after, the mix is more balanced while the hits stay on-topic (nursing/healthcare bios).

In [4]:
# inspect one query: top-5 results before vs after (relevance stays, gender mix evens out)
qi = 0  # "a registered nurse"
b = topk(Q[qi], Xdoc, 5); a = topk(Q_d[qi], Xdoc_d, 5)
lab = lambda g: 'F' if g == female else 'M'
print('QUERY:', queries[qi])
print('\nBEFORE erasure (top 5):')
for j in b: print(f'  [{lab(doc_gender[j])}] {doc_texts[j][:90]}')
print('\nAFTER erasure (top 5):')
for j in a: print(f'  [{lab(doc_gender[j])}] {doc_texts[j][:90]}')

QUERY: a registered nurse

BEFORE erasure (top 5):
  [F] Her clinical focus has been in the perioperative setting, which includes admitting and dis
  [M] He has an interest in nursing workforce issues and the relationship with nursing outcomes 
  [F] She earned her bachelor's degree in nursing at the University of Santo Tomas in the Philip
  [F] Andrea is a former President of the Royal College of Nursing and is a project manager at A
  [F] She joined the SHR in 2011 from the University Health Network (UHN) in Toronto where she s

AFTER erasure (top 5):
  [M] He has an interest in nursing workforce issues and the relationship with nursing outcomes 
  [F] Her clinical focus has been in the perioperative setting, which includes admitting and dis
  [M] He has worked as a nurse in the acute hospital in the ICU, Surgical, Ortho/Spine Units, fo
  [F] She earned her bachelor's degree in nursing at the University of Santo Tomas in the Philip
  [F] Andrea is a former President of the Royal Coll

## 4. Reading it

Erasing gender removes the gendered component that retrieval was keying on, so the top-k becomes less gender-skewed **without** a separate re-ranking step — the fix lives in the embedding space. Topical relevance is preserved because only the gender direction is removed. Swap the concept (e.g. `expand='auto'`) to debias retrieval for other attributes.